# CH06 從 Hugging Face 上下載模型、轉檔以及量化

## 下載模型、轉成 GGUF 格式和模型量化 (以國科會 TAIDE 模型為例)

### 1️⃣ 安裝 llama.cpp

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 1469, done.
remote: Counting objects: 100% (1469/1469), done.
remote: Compressing objects: 100% (1141/1141), done.
remote: Total 1469 (delta 317), reused 1036 (delta 279), pack-reused 0 (from 0)
Receiving objects: 100% (1469/1469), 23.23 MiB | 5.54 MiB/s, done.
Resolving deltas: 100% (317/317), done.
Updating files: 100% (1323/1323), done.


### 2️⃣ 確認授權並輸入 HUGGINGFACE_TOKEN
1. 請進入以下網址並登入帳號後, 確認使用條款

   https://huggingface.co/taide/Llama-3.1-TAIDE-LX-8B-Chat/tree/main

2. 點擊右上角的個人頭像, 選擇 **Access Tokens**

3. 點擊 **+Create new token**, Token type 選擇 **Read**, 並創建一個新的 Token

In [ ]:
from huggingface_hub import login
import getpass

HF_TOKEN = getpass.getpass("輸入你的 token: ")
login(HF_TOKEN)

輸入你的 token: ··········


### 3️⃣ 下載 safetensors 模型

In [ ]:
from huggingface_hub import snapshot_download

model_name = "taide-8b"
repo_id = "taide/Llama-3.1-TAIDE-LX-8B-Chat"

snapshot_download(
    repo_id = repo_id,
    local_dir = model_name,
    token=True
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:980: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

README_en.md:   0%|          | 0.00/25.3k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.5M [00:00<?, ?B/s]

'/content/taide-8b'

**如果要量化其他模型的話, 如果模型資料夾中已經包含 tokenizer.model 檔案就不用執行  4️⃣、5️⃣ 儲存格**

### 4️⃣ 改寫 convert_hf_to_gguf_update.py 檔案
在 llama.cpp 的轉檔流程裡，會使用 tokenizer.model 判斷採用哪類的 tokenizer,
如果模型資料夾只有 tokenizer.json, 則要在 convert_hf_to_gguf_update.py 的 models 中加入以下這行：
```
{"name": "model_name", "tokt": TOKENIZER_TYPE.BPE, "repo": "https://huggingface.co/taide/Llama-3.1-TAIDE-LX-8B-Chat", },
```


In [ ]:
from pathlib import Path, re

file_path = Path("/content/llama.cpp/convert_hf_to_gguf_update.py")

# 取出檔案
txt = file_path.read_text()

# 新的 models 區塊
models_block = f'''models = [
    {{"name": "{model_name}", "tokt": TOKENIZER_TYPE.BPE, "repo": "https://huggingface.co/{repo_id}", }},
]
'''

# 原先的 models 中已有部分模型, 若沒有權限會發生錯誤, 所以直接覆蓋整段 models = [...]
txt = re.sub(r"models\s*=\s*\[[\s\S]*?\]", models_block, txt, count=1)

file_path.write_text(txt)
print("已改寫 models 清單")

已改寫 models 清單


### 5️⃣ 擴充雜湊表, 以辨別 TAIDE 的 BPE tokenizer
執行後會改寫 convert_hf_to_gguf.py 的 ```get_vocab_base_pre()``` 函式

In [ ]:
%cd /content/llama.cpp
!python /content/llama.cpp/convert_hf_to_gguf_update.py $HF_TOKEN
%cd ..

/content/llama.cpp
python3 convert_hf_to_gguf.py models/tokenizers/taide-8b/ --outfile models/ggml-vocab-taide-8b.gguf --vocab-only
/content


### 6️⃣ 轉檔成 GGUF 格式
接下來就可以透過 convert_hf_to_gguf.py 來進行轉檔了,
執行後在檔案區可以看到 taide-8b.gguf


In [ ]:
output_gguf = f"{model_name}.gguf" # 輸出的檔案名稱

!python llama.cpp/convert_hf_to_gguf.py $model_name \
        --outfile $output_gguf

INFO:hf-to-gguf:Loading model: taide-8b
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {4096, 188256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {14336, 4096}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {4096, 14336}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {4096}
INFO:hf-to-gguf:blk.0.attn_k.w

### 7️⃣ 使用 CMake 產生編譯檔, 用來量化二進位檔案

In [ ]:
%cd /content/llama.cpp

# 用 CMake 產生編譯檔
!cmake -B build -DCMAKE_BUILD_TYPE=Release
!cmake --build build --target llama-quantize -j

# 確認可執行檔
!ls -lh build/bin/llama-quantize
%cd ..

/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (found version "4.5")
-- Found OpenMP: TRUE 

### 8️⃣ 量化模型

In [ ]:
quant_type = "Q4_K_M"
quant_out  = f"{model_name}-{quant_type}.gguf"

!./llama.cpp/build/bin/llama-quantize $output_gguf $quant_out $quant_type

main: build = 1 (fb1cab2)
main: built with cc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0 for x86_64-linux-gnu
main: quantizing '/content/taide-8b.gguf' to '/content/taide-8b-q4_k_m.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 30 key-value pairs and 292 tensors from /content/taide-8b.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Taide 8b
llama_model_loader: - kv   3:                           general.basename str              = taide
llama_model_loader: - kv   4:                         general.size_label str              = 8B
llama_model_loader: - kv   5:                            general.license str            

### 9️⃣ 撰寫 Modelfile
將下載好的 GGUF 檔案和 Modelfile 放在同一個資料夾, Modelfile 輸入以下內容 (由於 TAIDE 的基礎模型為 Llama3.1, 偷用它的格式)：
```
FROM ./taide-8b-Q4_K_M.gguf

TEMPLATE """
{{- if or .System .Tools }}<|start_header_id|>system<|end_header_id|>
{{- if .System }}

{{ .System }}
{{- end }}
{{- if .Tools }}

Cutting Knowledge Date: December 2023

When you receive a tool call response, use the output to format an answer to the orginal user question.

You are a helpful assistant with tool calling capabilities.
{{- end }}<|eot_id|>
{{- end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if eq .Role "user" }}<|start_header_id|>user<|end_header_id|>
{{- if and $.Tools $last }}

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{{ range $.Tools }}
{{- . }}
{{ end }}
Question: {{ .Content }}<|eot_id|>
{{- else }}

{{ .Content }}<|eot_id|>
{{- end }}{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- else if eq .Role "assistant" }}<|start_header_id|>assistant<|end_header_id|>
{{- if .ToolCalls }}
{{ range .ToolCalls }}
{"name": "{{ .Function.Name }}", "parameters": {{ .Function.Arguments }}}{{ end }}
{{- else }}

{{ .Content }}
{{- end }}{{ if not $last }}<|eot_id|>{{ end }}
{{- else if eq .Role "tool" }}<|start_header_id|>ipython<|end_header_id|>

{{ .Content }}<|eot_id|>{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- end }}
{{- end }}
"""

PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER stop "<|eot_id|>"
```